# Fine-tune PhoBERT phân loại bình luận (Showbiz / Giáo dục)

**Chuẩn bị:** file `phobert_package.zip` (gồm code `ml/phobert/*.py` + 2 file dữ liệu `verify_*_5k_dedup.xlsx`).

**Các bước:**
1. Menu **Runtime → Change runtime type → T4 GPU** (bắt buộc GPU, CPU sẽ rất lâu).
2. Chạy lần lượt từng ô từ trên xuống. Ô số 3 sẽ hỏi upload file zip.
3. Chạy xong, ô cuối tải về `phobert_results.zip` → gửi lại file này.

Thời gian ước tính trên T4: khoảng 10–20 phút mỗi lĩnh vực (tối đa 10 epoch, có dừng sớm).

Notebook chỉ gọi code có sẵn; **không cần sửa code**, chỉ chỉnh ô "Cấu hình" nếu được dặn.

## 1. Kiểm tra GPU

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ CHƯA CÓ GPU: vào Runtime → Change runtime type → chọn T4 GPU rồi chạy lại ô này.")

## 2. Cài thư viện

In [ ]:
!pip install -q "transformers>=4.40" "underthesea>=6.8" openpyxl scikit-learn pandas

## 3. Upload `phobert_package.zip` và giải nén

In [ ]:
import os, zipfile
from google.colab import files

uploaded = files.upload()                     # chọn phobert_package.zip
zip_name = next(n for n in uploaded if n.endswith(".zip"))
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content/pkg")
WORK = "/content/pkg/phobert"
DATA_DIR = "/content/pkg/data"
os.chdir(WORK)
print("Code:", sorted(os.listdir(WORK)))
print("Dữ liệu:", sorted(os.listdir(DATA_DIR)))

## 4. Cấu hình

- `DOMAINS`: lĩnh vực cần train (mặc định cả hai).
- `RUN_NO_SYNTHETIC = True`: train thêm một bản **không** dùng câu sinh tăng cường để so sánh (gấp đôi thời gian).
- Hết bộ nhớ GPU (lỗi `CUDA out of memory`) thì giảm `BATCH_SIZE` xuống 8.

In [ ]:
DOMAINS = ["showbiz", "education"]
EPOCHS = 10
BATCH_SIZE = 16
LR = 2e-5
MAX_LENGTH = 256
RUN_NO_SYNTHETIC = False
OUT = "/content/phobert_checkpoints"

## 5. Fine-tune

In [ ]:
runs = [(d, False) for d in DOMAINS] + ([(d, True) for d in DOMAINS] if RUN_NO_SYNTHETIC else [])
for domain, no_syn in runs:
    out_dir = f"{OUT}_no_synthetic" if no_syn else OUT
    data = f"{DATA_DIR}/verify_{domain}_5k_dedup.xlsx"
    flag = "--no_synthetic" if no_syn else ""
    print(); print(f"========== {domain} | {'KHÔNG' if no_syn else 'CÓ'} câu sinh ==========")
    !python phobert_finetune_colab.py --domain {domain} --data_path "{data}" --output_dir {out_dir} --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --max_length {MAX_LENGTH} {flag}

## 6. Xem kết quả

In [ ]:
import json, glob
import pandas as pd

rows = []
for path in sorted(glob.glob("/content/phobert_checkpoints*/*/test_results.json")):
    r = json.load(open(path, encoding="utf-8"))
    run = path.split("/")[-3].replace("phobert_checkpoints", "") or "có câu sinh"
    domain = path.split("/")[-2]
    rows.append({"lĩnh vực": domain, "bản": run, "test F1 macro": round(r["test_f1_macro"], 4),
                 "best val F1": round(r["best_val_f1"], 4), "n_test": r["n_test"], "epoch": r["epochs_run"]})
    print(); print(f"### {domain} ({run})")
    rep = pd.DataFrame(r["classification_report"]).T.round(3)
    display(rep)
    print("Ma trận nhầm lẫn (hàng = nhãn đúng, cột = dự đoán):")
    display(pd.DataFrame(r["confusion_matrix"], index=r["label_order"], columns=r["label_order"]))
display(pd.DataFrame(rows))

## 7. Tải kết quả về

Gói `phobert_results.zip` gồm cho mỗi lĩnh vực: `best_model.pt` (~540MB/lĩnh vực), `config.json`, `test_results.json`, `split_manifest.json`.

File lớn: nếu tải về chậm/lỗi, chạy ô 7b để lưu vào Google Drive thay vì tải.

In [ ]:
import os, glob, shutil
# Gom tất cả thư mục checkpoint vào một zip
os.makedirs("/content/_results", exist_ok=True)
for d in glob.glob("/content/phobert_checkpoints*"):
    shutil.copytree(d, f"/content/_results/{os.path.basename(d)}", dirs_exist_ok=True)
shutil.make_archive("/content/phobert_results", "zip", "/content/_results")
print("Kích thước:", round(os.path.getsize("/content/phobert_results.zip") / 1e6), "MB")
files.download("/content/phobert_results.zip")

### 7b. (Tuỳ chọn) Lưu vào Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
shutil.copy("/content/phobert_results.zip", "/content/drive/MyDrive/phobert_results.zip")
print("Đã lưu: MyDrive/phobert_results.zip")

## 8. (Tuỳ chọn) Gán nhãn hàng loạt bằng model vừa train

Upload file csv/xlsx có cột `text`, rồi sửa `INPUT` bên dưới.

In [ ]:
DOMAIN = "showbiz"
INPUT = "/content/comments.csv"
!python phobert-batch-inference.py --model_dir {OUT}/{DOMAIN} --input_path "{INPUT}" --output_path /content/predictions_{DOMAIN}.csv